# 05 · Overfitting and Regularisation
NumPy for the model · matplotlib for graphs · Runs on Google Colab

1. Simple example: fitting curves to 10 noisy points
2. Overfitting in our network + the validation set
3. Early stopping
4. L2 regularisation (weight decay)
5. Dropout
6. Real use case: digits with wrong labels
7. Experiments

## Setup: the same functions, with two small hooks (dropout in `forward_pass`, L2 in `backward_pass`)

In [ ]:
import io, urllib.request
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)
np.seterr(all="ignore")

def add_x0(X):
    return np.hstack([-np.ones((len(X), 1)), X])

def new_net(sizes):
    return [rng.normal(0, np.sqrt(2 / a), (a + 1, b)) for a, b in zip(sizes[:-1], sizes[1:])]   # He start (ReLU)

def forward_pass(X0, W, p_drop=0.0):
    H, cache = X0, []
    for w in W[:-1]:
        S = H @ w
        mask = (rng.random(S.shape) >= p_drop) / (1 - p_drop)      # dropout: switch off, scale survivors (all 1s if p_drop = 0)
        cache.append((H, S, mask))
        H = add_x0(np.maximum(0, S) * mask)
    cache.append((H, None, None))
    S = H @ W[-1]
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True), cache

def compute_error(Y, P):
    return Y - P

def backward_pass(W, error, cache, lam=0.0):
    delta, grads = error, [None] * len(W)
    for l in range(len(W) - 1, -1, -1):
        grads[l] = -cache[l][0].T @ delta / len(delta)
        grads[l][1:] += 2 * lam * W[l][1:]                         # L2: slope of λ·w² (thresholds in row 0 are left alone)
        if l > 0:
            delta = (delta @ W[l][1:].T) * np.where(cache[l - 1][1] > 0, 1.0, 0.0) * cache[l - 1][2]
    return grads

def adam_step(W, grads, state, lr):
    state["t"] += 1
    t, new = state["t"], []
    for k, (w, g) in enumerate(zip(W, grads)):
        state["m"][k] = 0.9 * state["m"][k] + 0.1 * g
        state["s"][k] = 0.999 * state["s"][k] + 0.001 * g ** 2
        new.append(w - lr * (state["m"][k] / (1 - 0.9 ** t)) / (np.sqrt(state["s"][k] / (1 - 0.999 ** t)) + 1e-8))
    return new

def loss(W, X0, y):
    P = forward_pass(X0, W)[0]
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-12))

def accuracy(W, X0, y):
    return 100 * (forward_pass(X0, W)[0].argmax(axis=1) == y).mean()

def train(X0, y, W, lr, epochs, batch_size, lam=0.0, p_drop=0.0, X_val=None, y_val=None, patience=None):
    # returns the final (or best, with patience) weights and the train/validation loss per epoch
    Y = np.eye(W[-1].shape[1])[y]
    state = {"t": 0, "m": [np.zeros_like(w) for w in W], "s": [np.zeros_like(w) for w in W]}
    history = {"train": [], "val": [], "best_epoch": epochs}
    best_W, best_val, wait = W, np.inf, 0
    for epoch in range(1, epochs + 1):
        idx = rng.permutation(len(X0))
        for s in range(0, len(X0), batch_size):
            batch = idx[s:s + batch_size]
            P, cache = forward_pass(X0[batch], W, p_drop)
            W = adam_step(W, backward_pass(W, compute_error(Y[batch], P), cache, lam), state, lr)
        history["train"].append(loss(W, X0, y))
        if X_val is not None:
            history["val"].append(loss(W, X_val, y_val))
            if patience is not None:                               # early stopping
                if history["val"][-1] < best_val:
                    best_W, best_val, wait, history["best_epoch"] = W, history["val"][-1], 0, epoch
                else:
                    wait += 1
                    if wait >= patience:
                        return best_W, history
    return (best_W if patience is not None else W), history

---
# Part 1: simple example — fitting curves to 10 noisy points
True curve: $y = \sin(3x)$, plus noise. Fit polynomials of degree 1, 3 and 9.

In [ ]:
x_pts = np.linspace(-1, 1, 10)                                    # 10 evenly spaced training points
y_pts = np.sin(3 * x_pts) + rng.normal(0, 0.2, 10)
x_new = rng.uniform(-1, 1, 200)                                   # fresh points from the same curve (test)
y_new = np.sin(3 * x_new) + rng.normal(0, 0.2, 200)

xs = np.linspace(-1, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, degree in zip(axes, [1, 3, 9]):
    coeffs = np.polyfit(x_pts, y_pts, degree)
    train_err = np.mean((np.polyval(coeffs, x_pts) - y_pts) ** 2)
    test_err = np.mean((np.polyval(coeffs, x_new) - y_new) ** 2)
    print(f"degree {degree}: training error {train_err:.4f} | test error {test_err:.4f}")
    ax.plot(xs, np.sin(3 * xs), "g--", label="true curve")
    ax.plot(xs, np.polyval(coeffs, xs), "crimson", label=f"degree {degree} fit")
    ax.scatter(x_pts, y_pts, c="k", zorder=3, label="10 training points")
    ax.set(ylim=(-2, 2), title=f"degree {degree}: train {train_err:.3f}, test {test_err:.3f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Degree 1: too simple (**underfitting**). Degree 3: good. Degree 9: passes through every training point (training error ≈ 0) but swings wildly between them: it memorised the noise (**overfitting**).

---
# Part 2: overfitting in our network
Two noisy "moons", only **100 training points**, big network `[2, 128, 128, 2]`, 500 epochs.

**Validation set:** a second, separate set of points used to *choose settings* (when to stop, how much regularisation). The **test** set is only looked at at the very end.

In [ ]:
def make_moons(n, noise=0.3):
    t, k = rng.uniform(0, np.pi, n), rng.integers(0, 2, n)
    X = np.where(k[:, None] == 0, np.c_[np.cos(t), np.sin(t)], np.c_[1 - np.cos(t), 0.5 - np.sin(t)])
    return X + rng.normal(0, noise, (n, 2)), k

Xm_tr, ym_tr = make_moons(100)
Xm_va, ym_va = make_moons(100)
Xm_te, ym_te = make_moons(2000)
Xm_tr0, Xm_va0, Xm_te0 = add_x0(Xm_tr), add_x0(Xm_va), add_x0(Xm_te)
MOON_SIZES = [2, 128, 128, 2]

g1, g2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.7, 2.2, 200))
GRID0 = add_x0(np.c_[g1.ravel(), g2.ravel()])

def plot_boundary(ax, W, title):
    Z = forward_pass(GRID0, W)[0][:, 1].reshape(g1.shape)
    ax.contourf(g1, g2, Z, levels=[0, 0.5, 1], colors=["#f8d0d0", "#d0e0f4"])
    ax.contour(g1, g2, Z, levels=[0.5], colors="k", linewidths=1)
    ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=np.where(ym_tr == 1, "#377eb8", "#e41a1c"), edgecolors="k", s=20)
    ax.set(title=title, xticks=[], yticks=[])

W_plain, hist_plain = train(Xm_tr0, ym_tr, new_net(MOON_SIZES), 0.01, 500, 16, X_val=Xm_va0, y_val=ym_va)
print(f"No regularisation: train accuracy {accuracy(W_plain, Xm_tr0, ym_tr):.1f}% | test accuracy {accuracy(W_plain, Xm_te0, ym_te):.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_boundary(axes[0], W_plain, "No regularisation: the boundary bends around single noisy points")
axes[1].plot(hist_plain["train"], label="training loss")
axes[1].plot(hist_plain["val"], label="validation loss")
axes[1].set(xlabel="epoch", ylabel="cross-entropy", title="Training loss keeps falling, validation loss rises")
axes[1].legend()
plt.show()

---
# Part 3: early stopping
After each epoch check the validation loss; keep the best weights; stop when it hasn't improved for `patience` epochs (same idea as the "pocket" weights in the perceptron notebook).

In [ ]:
W_early, hist_early = train(Xm_tr0, ym_tr, new_net(MOON_SIZES), 0.01, 500, 16, X_val=Xm_va0, y_val=ym_va, patience=30)
best = hist_early["best_epoch"]
print(f"Stopped after {len(hist_early['val'])} epochs, best validation loss at epoch {best}")
print(f"Early stopping: train accuracy {accuracy(W_early, Xm_tr0, ym_tr):.1f}% | test accuracy {accuracy(W_early, Xm_te0, ym_te):.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_boundary(axes[0], W_early, f"Early stopping (best epoch {best})")
axes[1].plot(hist_plain["val"], color="lightgray", label="validation loss, no stopping")
axes[1].plot(hist_early["val"], label="validation loss")
axes[1].axvline(best - 1, color="crimson", ls="--", label=f"best epoch {best}")
axes[1].set(xlabel="epoch", ylabel="cross-entropy", title="Stop where the validation loss is lowest")
axes[1].legend()
plt.show()

---
# Part 4: L2 regularisation (weight decay)
Loss $+\;\lambda \sum w^2$. Its slope adds $2\lambda w$, so every step also **shrinks** each weight a little.

In [ ]:
w_demo, slope, lr, lam = 2.0, 0.5, 0.1, 0.1
print(f"Without L2: w = {w_demo} - {lr} × {slope} = {w_demo - lr * slope:.3f}")
print(f"With L2   : w = {w_demo} - {lr} × ({slope} + 2 × {lam} × {w_demo}) = {w_demo - lr * (slope + 2 * lam * w_demo):.3f}  (pulled towards 0)")

In [ ]:
lams = [0.0, 0.001, 0.01, 0.1]
fig, axes = plt.subplots(2, len(lams), figsize=(20, 8))
for k, lam in enumerate(lams):
    W_l2, _ = train(Xm_tr0, ym_tr, new_net(MOON_SIZES), 0.01, 500, 16, lam=lam)
    print(f"λ = {lam:<5}: train {accuracy(W_l2, Xm_tr0, ym_tr):5.1f}% | test {accuracy(W_l2, Xm_te0, ym_te):5.1f}% | average |w| {np.mean([np.abs(w).mean() for w in W_l2]):.3f}")
    plot_boundary(axes[0, k], W_l2, f"λ = {lam}")
    axes[1, k].hist(np.concatenate([w[1:].ravel() for w in W_l2]), bins=60, range=(-0.6, 0.6), color="steelblue")
    axes[1, k].set(title=f"weights, λ = {lam}", yticks=[])
plt.tight_layout()
plt.show()

Bigger λ ➜ smaller weights ➜ smoother boundary. Too big (λ = 0.1) ➜ too smooth (underfitting).

---
# Part 5: dropout
During training, switch off each hidden neuron with probability $p$ and scale the survivors by $1/(1-p)$ so the average stays the same. At test time nothing is switched off.

In [ ]:
h = np.array([0.8, 1.5, 0.3, 2.0, 1.1, 0.6])
p = 0.5
mask = (rng.random(h.shape) >= p) / (1 - p)
print(f"hidden outputs     : {h}")
print(f"mask (p = {p})      : {mask}   (0 = switched off, {1 / (1 - p):.0f} = kept and scaled)")
print(f"after dropout      : {h * mask}")
print(f"average before/after over 10,000 random masks: {h.mean():.3f} / {np.mean([(h * (rng.random(h.shape) >= p) / (1 - p)).mean() for _ in range(10000)]):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, p in zip(axes, [0.0, 0.2, 0.5]):
    W_do, _ = train(Xm_tr0, ym_tr, new_net(MOON_SIZES), 0.01, 500, 16, p_drop=p)
    print(f"dropout p = {p}: train {accuracy(W_do, Xm_tr0, ym_tr):5.1f}% | test {accuracy(W_do, Xm_te0, ym_te):5.1f}%")
    plot_boundary(ax, W_do, f"dropout p = {p}")
plt.tight_layout()
plt.show()

On the moons the points themselves are noisy, so about 90% is close to the best possible: regularisation mainly makes the boundary **smoother**, the test accuracy changes only a little. Next, a case where it changes a lot.

---
# Part 6: real use case — digits where 20% of the labels are wrong
Real datasets often contain mislabelled examples. A big network can **memorise** them.
1,000 training images, 500 validation images (also 20% wrong), the normal test set. Network `[64, 256, 256, 10]`, 100 epochs.

In [ ]:
def load_digits(name):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/" + name
    d = np.genfromtxt(io.StringIO(urllib.request.urlopen(url).read().decode()), delimiter=",").astype(int)
    return d[:, :64] / 16, d[:, 64]

D_x, D_y = load_digits("optdigits.tra")
T_x, T_y = load_digits("optdigits.tes")
order = rng.permutation(len(D_x))
tr, va = order[:1000], order[1000:1500]

def corrupt(y, share):
    y = y.copy()
    wrong = rng.random(len(y)) < share
    y[wrong] = (y[wrong] + rng.integers(1, 10, wrong.sum())) % 10    # change to a different digit
    return y

Xd_tr0, Xd_va0, Xd_te0 = add_x0(D_x[tr]), add_x0(D_x[va]), add_x0(T_x)
yd_noisy, yd_va_noisy = corrupt(D_y[tr], 0.2), corrupt(D_y[va], 0.2)
print(f"Wrong labels in training: {(yd_noisy != D_y[tr]).sum()} of {len(tr)}")

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, i in zip(axes, np.where(yd_noisy != D_y[tr])[0][:8]):
    ax.imshow(D_x[tr][i].reshape(8, 8), cmap="gray_r")
    ax.set(title=f"labelled {yd_noisy[i]}", xticks=[], yticks=[])
plt.show()

In [21]:
DIGIT_SIZES = [64, 256, 256, 10]
methods = {"none": {}, "early stopping": {"patience": 10}, "L2 (λ = 0.01)": {"lam": 0.01},
           "dropout (p = 0.5)": {"p_drop": 0.5}, "early + dropout": {"patience": 10, "p_drop": 0.5}}

for labels, y_tr, y_va in [("20% wrong labels", yd_noisy, yd_va_noisy), ("clean labels", D_y[tr], D_y[va])]:
    print(f"\n{labels}")
    print(f"{'method':>18} | {'train acc':>9} | {'test acc':>8} | {'test loss':>9} | epochs")
    print("-" * 64)
    for name, kw in methods.items():
        W_d, hist = train(Xd_tr0, y_tr, new_net(DIGIT_SIZES), 0.001, 100, 32, X_val=Xd_va0, y_val=y_va, **kw)
        print(f"{name:>18} | {accuracy(W_d, Xd_tr0, y_tr):8.1f}% | {accuracy(W_d, Xd_te0, T_y):7.1f}% | "
              f"{loss(W_d, Xd_te0, T_y):9.3f} | {hist['best_epoch']}")

 dropout (p = 0.5) |    100.0% |    95.9% |     0.162 | 100
   early + dropout |    100.0% |    95.4% |     0.143 | 58


With wrong labels, the plain network reaches ~100% on its (partly wrong) training labels by memorising them, and test accuracy suffers. All three methods stop that. With clean labels, the differences are small.

---
# Part 7: Experiments (digits, 20% wrong labels)
## 1. Choosing λ and p with the validation set

In [ ]:
print(f"{'setting':>16} | {'validation acc':>14} | {'test acc':>8}")
print("-" * 46)
for name, kw in [(f"L2 λ = {lam}", {"lam": lam}) for lam in [0.0001, 0.001, 0.01, 0.05]] + \
                [(f"dropout p = {p}", {"p_drop": p}) for p in [0.2, 0.5, 0.7]]:
    W_s, _ = train(Xd_tr0, yd_noisy, new_net(DIGIT_SIZES), 0.001, 100, 32, **kw)
    print(f"{name:>16} | {accuracy(W_s, Xd_va0, yd_va_noisy):13.1f}% | {accuracy(W_s, Xd_te0, T_y):7.1f}%")

Validation accuracy looks low because 20% of the validation labels are wrong too; only the **ranking** matters. Pick the setting with the best validation accuracy, then report its test accuracy once.

## 2. More data (clean labels, no regularisation)

In [ ]:
sizes, accs = [100, 300, 1000, 3823], []
for n in sizes:
    pick = order[:n]
    W_n, _ = train(add_x0(D_x[pick]), D_y[pick], new_net(DIGIT_SIZES), 0.001, max(20, 30000 // n), 32)
    accs.append(accuracy(W_n, Xd_te0, T_y))
    print(f"{n:>5} training images: test accuracy {accs[-1]:.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(sizes, accs, "o-")
ax.set(xscale="log", xticks=sizes, xticklabels=sizes, xlabel="training images", ylabel="test accuracy (%)",
       title="More data is the strongest regulariser")
plt.show()

This completes the core deep-network ideas. Next: our own **mini-framework** (all of this as reusable classes), then **TensorFlow**, **Keras** and **PyTorch**.